# Day 1 — Naive RAG from Primitives

**Goal:** build an end-to-end RAG pipeline with *no framework magic*. Every
line is yours. By the end of today, you can point to every step and explain
what it does.

We'll use:
- **Plain Python** for ingestion and chunking
- **sentence-transformers** (free, local) AND **OpenAI embeddings** (paid) so we can compare
- **Chroma** as the vector store
- **Anthropic Claude** for generation

No LangChain. No LlamaIndex. You'll meet them later — first, you need to know
what they're abstracting.

## 1. Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from pathlib import Path
from utils import (
    get_llm_client, get_chroma_client,
    Chunk, make_chunk_id, add_chunks,
)

client, generate = get_llm_client()
chroma = get_chroma_client("./chroma_db")
print("Setup OK")

## 2. Load the clean corpus

Day 0 downloaded 5 PEPs to `./datasets/day1/`. Read them as-is.

In [ ]:
docs = []
for path in sorted(Path("./datasets/day1").glob("*.md")):
    text = path.read_text()
    docs.append({"source": path.name, "text": text})
    print(f"  {path.name}: {len(text):,} chars")

print(f"\nLoaded {len(docs)} documents.")

## 3. Chunking — the naive version

Split each doc into ~500-token chunks with ~50-token overlap. We'll use a
simple character-based splitter: 1 token ≈ 4 chars for English, so 500 tokens
≈ 2000 chars.

**This chunking is bad.** It doesn't respect sentence or paragraph boundaries.
That's the point — we want to see it fail so tomorrow's improvements are
motivated.

In [ ]:
def naive_split(text: str, chunk_size: int = 2000, overlap: int = 200) -> list[str]:
    chunks = []
    i = 0
    while i < len(text):
        chunks.append(text[i : i + chunk_size])
        i += chunk_size - overlap
    return chunks


all_chunks: list[Chunk] = []
for doc in docs:
    pieces = naive_split(doc["text"])
    for idx, piece in enumerate(pieces):
        all_chunks.append(Chunk(
            id=make_chunk_id(doc["source"], idx, piece),
            text=piece,
            source=doc["source"],
            extra={"chunk_idx": idx, "total_chunks": len(pieces)},
        ))

print(f"Created {len(all_chunks)} chunks from {len(docs)} documents.")
print(f"Sample chunk:\n---\n{all_chunks[0].text[:300]}...\n---")

## 4. Embed with sentence-transformers (free, local)

MiniLM-L6-v2 is tiny (80 MB), fast on CPU, and good enough for most prototypes.
First call downloads the model.

In [ ]:
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = [c.text for c in all_chunks]
st_embeddings = st_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
print(f"\nEmbedded {len(texts)} chunks → shape {st_embeddings.shape}")

## 5. Store in Chroma (collection 1: local embeddings)

In [ ]:
# Reset so re-running this notebook is clean
try:
    chroma.delete_collection("day1_local")
except Exception:
    pass

coll_local = chroma.create_collection(
    name="day1_local",
    # We're providing embeddings explicitly, so tell Chroma not to embed itself
    metadata={"embedding": "all-MiniLM-L6-v2"},
)

add_chunks(
    coll_local,
    all_chunks,
    embeddings=st_embeddings.tolist()
)
print(f"Collection 'day1_local' has {coll_local.count()} chunks.")

## 6. Embed with OpenAI (paid, stronger)

Only run if you have `OPENAI_API_KEY` set. Comment out if not.

In [ ]:
import os

USE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))

if USE_OPENAI:
    from openai import OpenAI
    oai = OpenAI()

    # Batch for API efficiency
    def embed_openai(texts: list[str], model: str = "text-embedding-3-small") -> list[list[float]]:
        # The API handles batching up to a limit; for small corpora just call once
        resp = oai.embeddings.create(model=model, input=texts)
        return [d.embedding for d in resp.data]

    oai_embeddings = embed_openai([c.text for c in all_chunks])
    print(f"OpenAI embeddings: {len(oai_embeddings)} vectors of dim {len(oai_embeddings[0])}")

    try:
        chroma.delete_collection("day1_openai")
    except Exception:
        pass
    coll_openai = chroma.create_collection(
        name="day1_openai",
        metadata={"embedding": "text-embedding-3-small"},
    )
    add_chunks(coll_openai, all_chunks, embeddings=oai_embeddings)
    print(f"Collection 'day1_openai' has {coll_openai.count()} chunks.")
else:
    print("Skipping OpenAI embeddings (no API key). Day 1 works fine with only local.")
    coll_openai = None

## 7. The retrieve function

Take a query, embed it, hit Chroma, return the top-k chunks.

In [ ]:
def retrieve(query: str, collection, embed_fn, k: int = 4) -> list[dict]:
    q_vec = embed_fn([query])[0]
    results = collection.query(query_embeddings=[q_vec], n_results=k)
    hits = []
    for i in range(len(results["ids"][0])):
        hits.append({
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
        })
    return hits


def embed_local(texts: list[str]) -> list[list[float]]:
    return st_model.encode(texts, convert_to_numpy=True).tolist()


# Smoke test
hits = retrieve("what is the zen of python", coll_local, embed_local, k=3)
for h in hits:
    print(f"[{h['distance']:.3f}] {h['metadata']['source']} — {h['text'][:100]}...")

## 8. The generate function

Format retrieved chunks into a prompt and call Claude. Notice we:
- tell Claude to ground in the provided sources
- ask for a short answer
- require it to cite the source file

These are the three hinges of RAG generation. Tune them for your use case.

In [ ]:
SYSTEM_PROMPT = """You are a precise technical assistant.

You will be given a user question and a set of source excerpts. Answer the \
question using ONLY the information in the sources. If the sources do not \
contain the answer, say so explicitly — do not invent information.

Keep your answer concise (3–5 sentences). At the end, list the source files \
you used on a line starting with "Sources: ".
"""


def format_sources(hits: list[dict]) -> str:
    lines = []
    for i, h in enumerate(hits, 1):
        lines.append(f"[Source {i}: {h['metadata']['source']}]")
        lines.append(h["text"])
        lines.append("")
    return "\n".join(lines)


def rag_answer(question: str, collection, embed_fn, k: int = 4) -> dict:
    hits = retrieve(question, collection, embed_fn, k=k)
    user = f"Sources:\n\n{format_sources(hits)}\n\nQuestion: {question}\n\nAnswer:"
    answer = generate(system=SYSTEM_PROMPT, user=user)
    return {"question": question, "answer": answer, "hits": hits}

## 9. Exercise 1.1 — Ask five questions

Five questions covering the corpus. Note how the retrieved sources change per
question — that's the "R" in RAG working.

In [ ]:
questions = [
    "What does PEP 8 recommend for the maximum line length?",
    "What is the Zen of Python's rule about readability?",
    "What does PEP 257 say about one-line docstrings?",
    "What are type hints in Python, according to PEP 484?",
    "How should imports be organized per PEP 8?",
]

for q in questions:
    print("=" * 70)
    print(f"Q: {q}")
    r = rag_answer(q, coll_local, embed_local)
    print(f"\nA: {r['answer']}")
    print(f"\n(Retrieved from: {', '.join(h['metadata']['source'] for h in r['hits'])})")
    print()

## 10. Exercise 1.2 — Conversation memory + the hallucination check

Add a simple memory buffer. Then ask a question the corpus *cannot* answer.
Watch what happens.

In [ ]:
class ChatRAG:
    def __init__(self, collection, embed_fn, k: int = 4, memory_turns: int = 3):
        self.collection = collection
        self.embed_fn = embed_fn
        self.k = k
        self.memory_turns = memory_turns
        self.history: list[tuple[str, str]] = []

    def ask(self, question: str) -> dict:
        # Simple approach: prepend last N turns into the question for retrieval
        # This helps when the user says "what about X?" as a follow-up
        recent = self.history[-self.memory_turns:]
        context_query = " ".join([f"{q} {a}" for q, a in recent] + [question])

        hits = retrieve(context_query, self.collection, self.embed_fn, k=self.k)

        # Include conversation history in the generation prompt
        history_text = "\n".join(f"User: {q}\nAssistant: {a}" for q, a in recent)
        user = (
            f"Prior conversation:\n{history_text}\n\n"
            f"Sources:\n{format_sources(hits)}\n\n"
            f"Current question: {question}\n\nAnswer:"
        )
        answer = generate(system=SYSTEM_PROMPT, user=user)
        self.history.append((question, answer))
        return {"question": question, "answer": answer, "hits": hits}


chat = ChatRAG(coll_local, embed_local)

print("Q1:", "What does PEP 8 say about indentation?")
r1 = chat.ask("What does PEP 8 say about indentation?")
print("A1:", r1["answer"])
print()
print("Q2 (follow-up):", "What about blank lines?")
r2 = chat.ask("What about blank lines?")
print("A2:", r2["answer"])
print()
# The gotcha: ask something not in the corpus
print("Q3 (not in corpus):", "What's the stock price of Nvidia?")
r3 = chat.ask("What's the stock price of Nvidia?")
print("A3:", r3["answer"])

## 11. Compare local vs OpenAI embeddings (if you ran both)

If you have an OpenAI key, rerun Q1 against both collections and compare.
The differences are often subtle on easy questions but big on technical
terminology or multi-hop queries.

In [ ]:
if coll_openai is not None:
    def embed_openai_fn(texts: list[str]) -> list[list[float]]:
        return embed_openai(texts)

    q = "What does PEP 484 say about generic types?"
    print("LOCAL (MiniLM):")
    r_local = rag_answer(q, coll_local, embed_local)
    print(r_local["answer"])
    print()
    print("OPENAI (text-embedding-3-small):")
    r_oai = rag_answer(q, coll_openai, embed_openai_fn)
    print(r_oai["answer"])

## 12. Reflection

By now you should notice:

1. **Chunks cut mid-sentence.** The naive splitter breaks text at arbitrary
   character offsets. Sometimes the retrieved chunk starts mid-sentence and
   Claude has to guess context. Tomorrow we fix this.

2. **No hierarchy.** We have no idea which section of a PEP a chunk came from.
   For the next 5 days we'll fix this with metadata + structured parsing.

3. **"What's Nvidia's stock price" — did Claude refuse or hallucinate?** If
   it refused, good — the system prompt did its job. If it made something up,
   note it; we'll add stronger guardrails on Day 5.

4. **Run the pipeline on the cursed PDFs tomorrow.** Your Day 1 pipeline
   works on clean markdown. Day 2 will break it.

Save your work and rest up.